# Figure S6: Protein vs RNA Correlation Dotplots

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys 
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import scanpy as sc 
from scipy.stats import pearsonr
from numpy.polynomial.polynomial import polyfit
sys.path.append('../utils')

import signature_heatmaps as signature_heatmaps
import factor_labels as factor_labels
import muon as mu


## Load Data

In [ ]:
data_dir = "<path to processed data>"

cite_6tf_path = os.path.join(data_dir, "cite_6tf_cleaned_revisions.h5mu")
cite_imgl_path = os.path.join(data_dir, "cite_imgl_cleaned_revisions.h5mu")
merged_6tf_path = os.path.join(data_dir, "adata_revisions_merged_6tf.h5ad")

In [ ]:
mdata_dict = {}
mdata_dict['cite_6tf'] = mu.read_h5mu(cite_6tf_path)
mdata_dict['cite_imgl'] = mu.read_h5mu(cite_imgl_path)

adata_dict = {}
adata_dict['merged_6tf'] = sc.read_h5ad(merged_6tf_path)
adata_dict['cite_6tf'] = mdata_dict['cite_6tf'].mod['rna'].copy()
adata_dict['cite_imgl'] = mdata_dict['cite_imgl'].mod['rna'].copy()

In [ ]:
signature_cols_ordered = ['homeostatic_score_ucell',
 'interferon_score_ucell',
 'chemokine_score_ucell',
 'antigen_presenting_score_ucell',
 'dam_score_ucell',
 'lipid_dam_score_ucell']

## Masking for analysis
to exclude ntc_g5 + foxk1_g2 + mixscale_cutoff >= 0

In [ ]:
guides_to_exclude = ['FOXK1_g2', 'non-targeting_g5']

adata_6tf_clean = adata_dict['merged_6tf'][~adata_dict['merged_6tf'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
adata_imgl_clean = adata_dict['cite_imgl'][~adata_dict['cite_imgl'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
adata_6tf_clean.shape, adata_imgl_clean.shape

In [ ]:
print(mdata_dict['cite_6tf'].shape, mdata_dict['cite_imgl'].shape)
mdata_6tf_clean = mdata_dict['cite_6tf'][~mdata_dict['cite_6tf'].mod['rna'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
mdata_imgl_clean = mdata_dict['cite_imgl'][~mdata_dict['cite_imgl'].mod['rna'].obs['guide'].isin(["non-targeting_g5", "FOXK1_g2"])]
mdata_6tf_clean.shape, mdata_imgl_clean.shape

In [ ]:
mixscale_col = "mixscale_score"

In [ ]:
adata_6tf_masked = adata_6tf_clean[adata_6tf_clean.obs[mixscale_col] >= 0].copy()
adata_imgl_masked = adata_imgl_clean[adata_imgl_clean.obs[mixscale_col] >= 0].copy()
adata_6tf_masked.shape, adata_imgl_masked.shape

In [ ]:
adata_masked_dict = {}
adata_masked_dict['iTF'] = adata_6tf_masked
adata_masked_dict['iMG'] = adata_imgl_masked

In [ ]:
mdata_masked_dict = {}
mdata_masked_dict['cite_6tf'] = mdata_6tf_clean
mdata_masked_dict['cite_imgl'] = mdata_imgl_clean

# Protein vs RNA Correlation

In [ ]:
def plot_protein_vs_rna_correlations(mudata, protein_name, layer=None, filter_zeros = False, to_plot=True):
    if layer:
        x = mudata.mod["rna"][:, protein_name].layers[layer].toarray().flatten()
        y = mudata.mod["prot"][:, protein_name].layers[layer].toarray().flatten()
    else:
        x = mudata.mod["rna"][:, protein_name].X.toarray().flatten()
        y = mudata.mod["prot"][:, protein_name].X.toarray().flatten()
        
    if filter_zeros:
        mask = (x != 0) & (y != 0)
        x_filtered = x[mask]
        y_filtered = y[mask] 
    else:
        x_filtered, y_filtered = x, y
    
    if (np.sum(x_filtered != 0) <= 3) or (np.sum(y_filtered != 0) <= 3):
        print(f"Skipping {protein_name}: less than 3 nonzero cells")
        return None, None, None
        
    a, b = polyfit(x_filtered, y_filtered, deg=1)
    corr_coeff, pval_coeff = pearsonr(x_filtered, y_filtered)   
    
    stats = {
        "protein_name": protein_name,
        "n_nonzero_rna": int(np.sum(x != 0)),
        "n_nonzero_prot": int(np.sum(y != 0)),
        "n_cells": len(x_filtered)
    }
    print(f"{protein_name}: {stats['n_cells']} cells, {stats['n_nonzero_rna']} nonzero RNA, {stats['n_nonzero_prot']} nonzero prot")
    
    if to_plot:
        mudata_plot = mudata[mask].copy() if filter_zeros else mudata
        mu.pl.scatter(mudata_plot, x=f"rna:{protein_name}", y=f"prot:{protein_name}", show=False, use_raw=False,
                      layers = layer)
        fig = plt.gcf()
        fig.set_dpi(300)
        ax = plt.gca()
        ax.plot(x_filtered, a + b * x_filtered, "-", color="red")
        ax.text(0.05, 0.95, f"r = {corr_coeff:.2f}", 
                transform=plt.gca().transAxes,
                fontsize=12, 
                verticalalignment='top',
                bbox=dict(boxstyle='round,pad=0.3', edgecolor='black', facecolor='white'))

        ax.set_xlabel(f"{protein_name} (RNA)")
        ax.set_ylabel(f"{protein_name} (Protein)")
        ax.set_title(protein_name, fontsize=20)
        plt.show()
    return corr_coeff, pval_coeff, stats

In [ ]:
def get_common_proteins(mdata,
                        prot_modality_label = 'prot',
                        rna_modality_label = 'rna'):
    rna_vals_for_prot = [p for p in mdata.mod[prot_modality_label].var.index if p in mdata.mod[rna_modality_label].var.index]
    rna_vals_for_prot_not = [p for p in mdata.mod[prot_modality_label].var.index if p not in mdata.mod[rna_modality_label].var.index]
    return rna_vals_for_prot, rna_vals_for_prot_not

In [ ]:
corr_dict = {}
for name, mdata in mdata_masked_dict.items():
    rna_vals_for_prot, _ = get_common_proteins(mdata)
    results = []
    skipped = []
    for p in rna_vals_for_prot:
        print(p)
        corr_coeff, pval_coeff, stats = plot_protein_vs_rna_correlations(mdata, p, filter_zeros=True, to_plot=False)
        if corr_coeff is None:
            skipped.append(p)
            continue
        stats["corr"] = corr_coeff
        stats["pval"] = pval_coeff
        stats["sig"] = pval_coeff <= 0.05
        results.append(stats)
    stats_df = pd.DataFrame(results)
    corr_dict[name] = stats_df


In [ ]:
def rna_protein_corr_dotplot(corr_df, figsize, savefig, n_cells_filter = 30, n_cells_display=False):
    fig, ax = plt.subplots(figsize=figsize, dpi=300)
    filtered = corr_df[corr_df['n_cells'] >= n_cells_filter].sort_values("corr").copy()
    if n_cells_display:
        filtered["label"] = filtered["protein_name"] + " (n=" + filtered["n_cells"].astype(str) + ")"
    else:
        filtered["label"] = filtered["protein_name"]
    filtered["significant"] = (filtered["sig"]).astype(int)

    edge_colors = ["black" for _ in filtered["significant"]]
    scatter = ax.scatter(filtered["label"], filtered["corr"], 
                     s=100, linewidths=1.5,
                     edgecolors=edge_colors,
                     facecolors=["black" if s > 0 else "none" for s in filtered["significant"]])
   
    ax.axhline(0, color="grey", linestyle="--")
    ax.set_ylabel("Pearson r")
    ax.set_xlabel("Proteins")
    ax.set_title("RNA-Protein Correlation")
    ax.set_xlim(-0.5, len(filtered) - 0.5)
    plt.xticks(rotation=90)
    
    plt.tight_layout()
    if savefig:
        fig.savefig(savefig)
    plt.show()

In [ ]:
fig_dir = "<path to figure directory>"
fig_dir_img = os.path.join(fig_dir, "iMG/final/")
fig_dir_itf = os.path.join(fig_dir, "iTF/final/")

In [ ]:
plt.rcParams["font.family"] = "Arial"

rna_protein_corr_dotplot(corr_dict['cite_6tf'], (25,6), 
                         os.path.join(fig_dir_itf, "itf_rna_prot_correlations_dotplot.svg"), 
                         n_cells_filter = 30, 
                         n_cells_display=False)

In [ ]:
rna_protein_corr_dotplot(corr_dict['cite_imgl'], (25,6), 
                         os.path.join(fig_dir_img, "img_rna_prot_correlations_dotplot.svg"), 
                         n_cells_filter = 30, 
                         n_cells_display=False)